# Notebook 06 — Height Gradient Analysis
**Part C — Vertical concentration gradient**

## What this notebook does
1. Assigns sampling heights to stations (0.91 m, 7.66 m, 15.3 m)  
2. Fits **power-law decay models** C = a × h⁻ᵇ  
3. Runs **Wilcoxon signed-rank tests** (ground vs rooftop, paired by date)  
4. Computes **Cohen's d** effect sizes  
5. Computes **rank-biserial correlation** r_rb  
6. Computes **Spearman ρ** (height vs concentration, all 64 observations)  
7. Computes **I/O penetration ratio** (Station 6 window / Station 5 rooftop)  
8. Produces all height gradient figures

## Sampling heights
| Station | Location | Height |
|---------|----------|--------|
| 1, 2, 3, 4 | Ground level | 0.91 m (3 ft) |
| 6 | Indoor window | 7.66 m (25.125 ft) |
| 5 | Rooftop | 15.3 m (50.25 ft) |

Sites 5 and 6 are on the **same building** → enables within-building paired comparison.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# COLAB SETUP — run this cell first if using Google Colab
# ═══════════════════════════════════════════════════════════════
import os, sys

# Option A: Clone the GitHub repo directly in Colab (recommended)
# !git clone https://github.com/Filza-coder/geoai-bioaerosol-prediction.git
# os.chdir('geoai-bioaerosol-prediction')

# Option B: Mount Google Drive and navigate to your folder
# from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/geoai-bioaerosol-prediction')

# Install dependencies
# !pip install openpyxl geopandas shapely pyproj scikit-learn shap seaborn -q

print('Current directory:', os.getcwd())
print('Python:', sys.version[:10])

In [ ]:
# ── Install (uncomment on Colab) ──────────────────────────────────
# !pip install matplotlib scipy numpy pandas -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('data/df_analysis.csv')
for col in ['Aspergillus_conc', 'Alternaria_conc']:
    df[col] = df[col].fillna(df[col].median())

# Assign sampling heights (feet converted to metres)
HEIGHT_M  = {1: 0.914, 2: 0.914, 3: 0.914, 4: 0.914,
             5: 15.316, 6: 7.658}   # 3ft, 50.25ft, 25.125ft
HEIGHT_FT = {1: 3, 2: 3, 3: 3, 4: 3, 5: 50.25, 6: 25.125}

df['height_m']  = df['station'].map(HEIGHT_M)
df['height_ft'] = df['station'].map(HEIGHT_FT)

TARGETS = {
    'pollen_conc':      ('Pollen',       '#378ADD'),
    'fungus_conc':      ('Total Fungus', '#D85A30'),
    'Aspergillus_conc': ('Aspergillus',  '#1D9E75'),
    'Alternaria_conc':  ('Alternaria',   '#BA7517'),
}

print('Heights assigned:')
for s, h in HEIGHT_M.items():
    print(f'  Station {s}: {HEIGHT_FT[s]} ft = {h} m')

## Step 1 — Prepare paired comparison data
Ground mean = daily average of Stations 1–4  
Rooftop     = Station 5  
Window      = Station 6  

Stations 5 and 6 are co-located on the same building, so we can do:  
- Ground vs Rooftop (independent of site differences)  
- Rooftop vs Window (true within-building height comparison)

In [ ]:
# Ground mean per date (average of stations 1–4)
ground_mean = (df[df['station'].isin([1,2,3,4])]
               .groupby('date')[[t for t in TARGETS]].mean())

# Rooftop (station 5) and window (station 6) indexed by date
roof   = df[df['station'] == 5].set_index('date')[[t for t in TARGETS]]
window = df[df['station'] == 6].set_index('date')[[t for t in TARGETS]]

# Find common dates for each comparison
common_gr = ground_mean.index.intersection(roof.index)
common_gw = ground_mean.index.intersection(window.index)
common_rw = roof.index.intersection(window.index)

print(f'Ground vs Roof  — matched dates: {len(common_gr)}: {list(common_gr)}')
print(f'Ground vs Window — matched dates: {len(common_gw)}')
print(f'Roof vs Window  — matched dates: {len(common_rw)}: {list(common_rw)}')

# Mean concentrations at each height level
heights_3 = np.array([0.914, 7.658, 15.316])
height_groups = {
    'Ground (0.9m)':  [1,2,3,4],
    'Window (7.7m)':  [6],
    'Roof (15.3m)':   [5],
}
print('\nMean concentrations by height:')
for label, stas in height_groups.items():
    sub = df[df['station'].isin(stas)]
    print(f'  {label}:')
    for tcol, (tlabel, _) in TARGETS.items():
        print(f'    {tlabel}: {sub[tcol].mean():.1f} grain/m³')

## Step 2 — Power-law decay fitting: C = a × h⁻ᵇ
Fit in log-log space: log(C) = log(a) − b × log(h)  
This is equivalent to linear regression of log(C) on log(h).

In [ ]:
print('Power-law decay fit: C = a × h^(-b)')
print(f'{"Target":20s}  {"a":>8}  {"b":>8}  {"R²":>6}')
print('-'*50)

decay_fits = {}
for tcol, (tlabel, col) in TARGETS.items():
    # Mean at each of the 3 heights
    means = []
    for stas in [[1,2,3,4], [6], [5]]:
        means.append(df[df['station'].isin(stas)][tcol].mean())
    means = np.array(means)

    # Log-log linear regression
    log_h = np.log(heights_3)
    log_c = np.log(means)
    slope, intercept, r, p, _ = stats.linregress(log_h, log_c)
    a  = np.exp(intercept)
    b  = -slope   # decay exponent (positive = decreasing with height)
    r2 = r**2

    decay_fits[tcol] = {'a': a, 'b': b, 'r2': r2, 'means': means}
    print(f'{tlabel:20s}  C = {a:.1f} × h^(-{b:.3f})   R² = {r2:.3f}')

## Step 3 — Wilcoxon signed-rank test + Effect sizes
### Wilcoxon signed-rank test
- Non-parametric equivalent of paired t-test  
- Appropriate for small n and non-normal data  
- Tests whether ground concentrations are consistently higher than rooftop

### Cohen's d
d = (mean₁ − mean₂) / pooled SD  
- Small: d < 0.2 | Medium: d ≈ 0.5 | **Large: d > 0.8**

### Rank-biserial correlation r_rb
- Derived from Wilcoxon statistic: r_rb = 1 − 2W / n(n+1)  
- Measures what proportion of paired comparisons favour ground > rooftop  
- r_rb = 1.0 means ground > rooftop on **every single** paired date

In [ ]:
print('Statistical tests: Ground vs Rooftop (paired by date)')
print(f'{"Target":20s}  {"Drop":>6}  {"Cohen d":>8}  {"r_rb":>6}  {"W":>6}  {"p":>8}')
print('-'*68)

test_results = {}
for tcol, (tlabel, col) in TARGETS.items():
    g = ground_mean.loc[common_gr, tcol].values
    r = roof.loc[common_gr, tcol].values

    pct_drop  = (g.mean() - r.mean()) / g.mean() * 100
    pooled_sd = np.sqrt((g.std()**2 + r.std()**2) / 2)
    cohens_d  = (g.mean() - r.mean()) / pooled_sd

    W, p_val  = stats.wilcoxon(g, r)
    n         = len(g)
    r_rb      = 1 - (2 * W) / (n * (n + 1))

    test_results[tcol] = {
        'pct_drop': pct_drop, 'cohens_d': cohens_d,
        'r_rb': r_rb, 'W': W, 'p': p_val,
        'g': g, 'r': r
    }
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    print(f'{tlabel:20s}  {pct_drop:>5.0f}%  {cohens_d:>8.3f}  '
          f'{r_rb:>6.3f}  {W:>6.0f}  {p_val:>8.4f} {sig}')

print()
print('Cohen d interpretation: small<0.2 | medium≈0.5 | large>0.8')
print('All d values > 0.8 = very large effect (height is a major determinant)')

## Figure 7 — Height gradient: box plots + power-law decay curves
Top row: box plots at three height levels with significance brackets  
Bottom row: power-law decay curve fitted to mean concentrations

In [ ]:
fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 4, hspace=0.45, wspace=0.38)

for i, (tcol, (tlabel, col)) in enumerate(TARGETS.items()):
    # ── Top row: box plots ─────────────────────────────────────
    ax_box = fig.add_subplot(gs[0, i])
    box_data = [df[df['station'].isin(stas)][tcol].dropna().values
                for stas in [[1,2,3,4], [6], [5]]]
    bp = ax_box.boxplot(box_data, patch_artist=True, widths=0.5,
                         medianprops=dict(color='black', lw=2))
    for patch in bp['boxes']:
        patch.set_facecolor(col); patch.set_alpha(0.75)

    ax_box.set_xticklabels(['Ground\n(0.9m)', 'Window\n(7.7m)', 'Roof\n(15.3m)'], fontsize=8)
    ax_box.set_ylabel('Concentration (grain/m³)', fontsize=9)
    ax_box.set_title(tlabel, fontsize=10, fontweight='bold', color=col)
    ax_box.spines[['top', 'right']].set_visible(False)

    # Significance bracket (ground vs roof)
    res = test_results[tcol]
    p   = res['p']
    sym = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    y_top = max([d.max() for d in box_data if len(d)]) * 1.08
    ax_box.annotate('', xy=(3, y_top), xytext=(1, y_top),
                    arrowprops=dict(arrowstyle='-', lw=1.2))
    ax_box.text(2, y_top * 1.02, f'{sym}\nd={res["cohens_d"]:.2f}',
                ha='center', fontsize=8)

    # ── Bottom row: power-law curve ────────────────────────────
    ax_dec = fig.add_subplot(gs[1, i])
    fit = decay_fits[tcol]
    h_range = np.linspace(0.5, 18, 200)
    ax_dec.plot(h_range, fit['a'] * h_range**(-fit['b']), color=col, lw=2.2,
                label=f"C = {fit['a']:.1f}·h$^{{-{fit['b']:.2f}}}$\nR²={fit['r2']:.2f}")
    ax_dec.scatter(heights_3, fit['means'], color=col, s=80, zorder=5,
                    edgecolors='white', linewidth=1)
    for h, m, lbl in zip(heights_3, fit['means'], ['Ground','Window','Roof']):
        ax_dec.annotate(f'{lbl}\n{m:.0f}', (h, m), xytext=(8, 5),
                         textcoords='offset points', fontsize=7.5, color='#333')
    ax_dec.set_xlabel('Height (m)', fontsize=9)
    ax_dec.set_ylabel('Mean conc. (grain/m³)', fontsize=9)
    ax_dec.set_title(f'{tlabel} — decay with height', fontsize=9)
    ax_dec.legend(fontsize=8, loc='upper right')
    ax_dec.set_xlim(0, 18)
    ax_dec.spines[['top', 'right']].set_visible(False)

fig.suptitle('Height gradient analysis\nTop: concentration by height | Bottom: power-law decay C = a·h⁻ᵇ',
             fontsize=11, y=1.01)
plt.savefig('fig_height_gradient.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: fig_height_gradient.png')

## Figure 8 — Spearman: height vs concentration (all 64 rows pooled)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4.5))
rng = np.random.default_rng(42)

for i, (tcol, (tlabel, col)) in enumerate(TARGETS.items()):
    ax = axes[i]
    sub = df[['height_m', tcol]].dropna()
    h  = sub['height_m'].values
    c  = sub[tcol].values
    rho, p = stats.spearmanr(h, c)

    # Jitter height slightly so overlapping points are visible
    hj = h + rng.uniform(-0.3, 0.3, len(h))
    ax.scatter(hj, c, color=col, alpha=0.65, s=40,
               edgecolors='white', linewidth=0.4)

    # Group mean diamonds
    for hh, label in zip([0.914, 7.658, 15.316], ['Ground','Window','Roof']):
        mn = df[df['height_m'] == hh][tcol].mean()
        ax.plot(hh, mn, 'D', color='black', ms=8, zorder=5)
        ax.text(hh + 0.4, mn, f'{mn:.0f}', fontsize=8, va='center')

    # Trend line
    xs = np.linspace(h.min(), h.max(), 50)
    z  = np.polyfit(h, c, 1)
    ax.plot(xs, np.poly1d(z)(xs), '--', color=col, lw=1.8, alpha=0.7)

    sym = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    ax.set_xlabel('Sampling height (m)', fontsize=9)
    ax.set_ylabel('Concentration (grain/m³)', fontsize=9)
    ax.set_title(f'{tlabel}\nSpearman ρ={rho:.2f} {sym}', fontsize=10, fontweight='bold')
    ax.set_xticks([0.914, 7.658, 15.316])
    ax.set_xticklabels(['0.9\n(Ground)', '7.7\n(Window)', '15.3\n(Roof)'], fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Spearman ρ: sampling height vs concentration (all 64 obs pooled)\n'
             '◆ = group mean  ** p<0.01  *** p<0.001',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('fig_height_spearman.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: fig_height_spearman.png')

## Figure 9 — I/O Penetration Ratio
Indoor/outdoor ratio = Station 6 (window, 7.7m) / Station 5 (rooftop, 15.3m)  
Both are on the same building → this is a true within-building height comparison.  
Ratio > 1 = more concentration near the window than at the rooftop.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
COLORS = {t: col for t, (_, col) in TARGETS.items()}

# Left: ratio over time for all species
ax = axes[0]
for tcol, (tlabel, col) in TARGETS.items():
    r_vals = roof.loc[common_rw, tcol].values
    w_vals = window.loc[common_rw, tcol].values
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(r_vals > 0, w_vals / r_vals, np.nan)
    dates_s = [d[5:] for d in common_rw]   # MM-DD format
    ax.plot(dates_s, ratio, marker='o', ms=5, lw=1.5,
             label=tlabel, color=col)

ax.axhline(1, color='black', lw=1, ls='--', alpha=0.5, label='Ratio = 1')
ax.set_xlabel('Date (MM-DD)', fontsize=10)
ax.set_ylabel('Concentration ratio (Window / Roof)', fontsize=10)
ax.set_title('I/O penetration ratio over time\nRatio > 1 = more at window than rooftop', fontsize=10)
ax.legend(fontsize=8); ax.spines[['top', 'right']].set_visible(False)
plt.setp(ax.get_xticklabels(), rotation=35, ha='right', fontsize=8)

# Right: mean ± SD bars
ax2 = axes[1]
ratio_means, ratio_stds, labels, colors = [], [], [], []
for tcol, (tlabel, col) in TARGETS.items():
    r_v = roof.loc[common_rw, tcol].values
    w_v = window.loc[common_rw, tcol].values
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(r_v > 0, w_v / r_v, np.nan)
    ratio = ratio[~np.isnan(ratio)]
    ratio_means.append(ratio.mean())
    ratio_stds.append(ratio.std())
    labels.append(tlabel)
    colors.append(col)

bars = ax2.bar(labels, ratio_means, color=colors, alpha=0.82, edgecolor='white', width=0.55)
ax2.errorbar(labels, ratio_means, yerr=ratio_stds,
              fmt='none', color='#333', capsize=5, lw=1.5)
ax2.axhline(1, color='black', lw=1, ls='--', alpha=0.5)
for bar, m in zip(bars, ratio_means):
    ax2.text(bar.get_x() + bar.get_width()/2, m + ratio_stds[bars.index(bar)] + 0.1,
              f'{m:.2f}', ha='center', fontsize=9, fontweight='bold')
ax2.set_ylabel('Mean I/O ratio ± SD', fontsize=10)
ax2.set_title('Mean penetration ratio\n(Window 7.7m / Roof 15.3m)', fontsize=10)
ax2.spines[['top', 'right']].set_visible(False)

fig.suptitle('Indoor/outdoor penetration analysis — Station 6 (window) vs Station 5 (rooftop)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('fig_io_penetration.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: fig_io_penetration.png')

In [ ]:
# ── Final summary table (matches Table 7 in paper) ────────────────
print('Table 7 — Height gradient summary (matches paper)')
print(f'{"Target":20s}  {"Decay model":20s}  {"R²":>4}  '
      f'{"Ground":>8}  {"Window":>8}  {"Roof":>8}  '
      f'{"Drop":>6}  {"Cohens_d":>8}  {"W-test p":>9}')
print('-'*105)

for tcol, (tlabel, col) in TARGETS.items():
    fit = decay_fits[tcol]
    res = test_results[tcol]
    means = fit['means']   # [ground, window, roof]

    # Spearman height vs concentration (all rows)
    sub    = df[['height_m', tcol]].dropna()
    rho, _ = stats.spearmanr(sub['height_m'], sub[tcol])
    sig_p  = '***' if res['p']<0.001 else '**' if res['p']<0.01 else '*' if res['p']<0.05 else 'ns'

    print(f'{tlabel:20s}  C={fit["a"]:.1f}·h^(-{fit["b"]:.3f}){" ":5}  '
          f'{fit["r2"]:>4.2f}  '
          f'{means[0]:>8.1f}  {means[1]:>8.1f}  {means[2]:>8.1f}  '
          f'{res["pct_drop"]:>5.0f}%  {res["cohens_d"]:>8.3f}  '
          f'{res["p"]:>8.4f} {sig_p}')